# Force density method with NumPy

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps

In [ ]:
import numpy as np
from compas.geometry import Line
from compas_notebook import Viewer
from compas_notebook.config import Config

## Net: five free nodes, four fixed nodes

In [ ]:
edges = [(0, 5), (1, 6), (2, 7), (3, 8), (0, 3), (0, 1), (1, 2), (2, 3), (3, 4), (0, 4), (1, 4), (2, 4)]
xyz_fixed = np.array([[-2, 0, 0], [0, -2, 1], [2, 0, 0], [0, 2, 1]], dtype=float)
nfree = 5
q = np.array([2.0] * 4 + [1.0] * 8)
p = np.zeros((nfree, 3))
p[:, 2] = -0.5

## Connectivity matrix

In [ ]:
C = np.zeros((len(edges), nfree + len(xyz_fixed)))
for j, (i, k) in enumerate(edges):
    C[j, i], C[j, k] = 1, -1
Cn, Cf = C[:, :nfree], C[:, nfree:]
Q = np.diag(q)

## Solve for the free nodes

In [ ]:
xyz_free = np.linalg.solve(Cn.T @ Q @ Cn, p - Cn.T @ Q @ Cf @ xyz_fixed)
xyz = np.vstack([xyz_free, xyz_fixed])
forces = q * np.linalg.norm(C @ xyz, axis=1)
xyz_free.round(3), forces.round(3)

## Scene

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [0, -6, 4]
config.view.camera.target = [0, 0, 0]
viewer = Viewer(config=config)
for i, k in edges:
    viewer.scene.add(Line(xyz[i], xyz[k]), color="#5552bb")
viewer.show()